# InternVL3 Hybrid Processor Test

**SAFE TESTING**: Tests the new hybrid processor (InternVL3 model + Llama pipeline) without affecting existing code.

**Architecture**: 
- InternVL3 model for accuracy
- Llama's proven processing pipeline for reliability
- ExtractionCleaner for 🧹 CLEANER CALLED output
- Working 82% prompts (reverted from simple format)

**Expected Results**: Should combine the best of both worlds - InternVL3 accuracy + Llama reliability.

In [1]:
# Setup and imports with proper environment detection
import sys
import os
from pathlib import Path
import time

# CRITICAL FIX: Detect environment and set correct paths
current_dir = os.getcwd()
if '/home/jovyan/nfs_share/tod' in current_dir:
    # AISandbox environment
    project_root = '/home/jovyan/nfs_share/tod/LMM_POC'
    deployment_env = "AISandbox"
    models_base = '/home/jovyan/nfs_share/models'
elif '/efs/shared' in current_dir:
    # EFS environment
    project_root = '/efs/shared/LMM_POC'
    deployment_env = "efs"
    models_base = '/efs/shared/PTM'
else:
    # Local development - use current directory
    project_root = str(Path.cwd())
    deployment_env = "local"
    models_base = f"{project_root}/models"

# Add project root to path
sys.path.insert(0, project_root)
os.chdir(project_root)

print(f"📍 Project root: {project_root}")
print(f"🌍 Detected environment: {deployment_env}")
print(f"🚀 Models base: {models_base}")
print(f"🐍 Python path: {sys.path[0]}")
print(f"📂 Working directory: {os.getcwd()}")

# CRITICAL: Override deployment configuration
os.environ['CURRENT_DEPLOYMENT'] = deployment_env
print(f"✅ Set CURRENT_DEPLOYMENT={deployment_env}")

📍 Project root: /home/jovyan/nfs_share/tod/LMM_POC
🌍 Detected environment: AISandbox
🚀 Models base: /home/jovyan/nfs_share/models
🐍 Python path: /home/jovyan/nfs_share/tod/LMM_POC
📂 Working directory: /home/jovyan/nfs_share/tod/LMM_POC
✅ Set CURRENT_DEPLOYMENT=AISandbox


In [2]:
# Import the hybrid processor
try:
    from models.document_aware_internvl3_hybrid_processor import DocumentAwareInternVL3HybridProcessor
    print("✅ InternVL3 Hybrid Processor imported successfully")
except ImportError as e:
    print(f"❌ Import error: {e}")
    print("💡 Check that the project root path is correct")
    raise

✅ InternVL3 Hybrid Processor imported successfully


In [3]:
# Configuration with model path validation
# Start with small field list for initial testing
TEST_FIELDS_SMALL = [
    "DOCUMENT_TYPE",
    "SUPPLIER_NAME", 
    "TOTAL_AMOUNT"
]

TEST_FIELDS_MEDIUM = [
    "DOCUMENT_TYPE",
    "SUPPLIER_NAME",
    "BUSINESS_ADDRESS",
    "TOTAL_AMOUNT",
    "GST_AMOUNT",
    "INVOICE_DATE"
]

# HARDCODED paths for sandbox testing
INTERNVL3_MODEL_PATH = '/home/jovyan/nfs_share/models/InternVL3-8B'
TEST_IMAGE_PATH = "/home/jovyan/nfs_share/tod/evaluation_data/image_001.png"

print(f"🎯 Small test fields: {TEST_FIELDS_SMALL}")
print(f"🎯 Medium test fields: {TEST_FIELDS_MEDIUM}")
print(f"🚀 InternVL3 model path: {INTERNVL3_MODEL_PATH}")
print(f"🖼️ Test image: {TEST_IMAGE_PATH}")

# Validate paths exist
from pathlib import Path
model_path = Path(INTERNVL3_MODEL_PATH)
if model_path.exists():
    print("✅ InternVL3 model path validated")
else:
    print(f"❌ CRITICAL: Model path does not exist: {INTERNVL3_MODEL_PATH}")

if Path(TEST_IMAGE_PATH).exists():
    print("✅ Test image found")
else:
    print("⚠️ Test image not found")

🎯 Small test fields: ['DOCUMENT_TYPE', 'SUPPLIER_NAME', 'TOTAL_AMOUNT']
🎯 Medium test fields: ['DOCUMENT_TYPE', 'SUPPLIER_NAME', 'BUSINESS_ADDRESS', 'TOTAL_AMOUNT', 'GST_AMOUNT', 'INVOICE_DATE']
🚀 InternVL3 model path: /home/jovyan/nfs_share/models/InternVL3-8B
🖼️ Test image: /home/jovyan/nfs_share/tod/evaluation_data/image_001.png
✅ InternVL3 model path validated
✅ Test image found


## Phase 1: Basic Initialization Test

In [4]:
# Test basic initialization (small field list) with enhanced error handling
print("🚀 Testing hybrid processor initialization...")

try:
    # Initialize hybrid processor with HARDCODED model path
    hybrid_processor = DocumentAwareInternVL3HybridProcessor(
        field_list=TEST_FIELDS_SMALL,
        model_path=INTERNVL3_MODEL_PATH,  # Pass hardcoded path directly
        debug=True  # Enable verbose output including 🧹 CLEANER CALLED
    )
    
    print("\n✅ HYBRID PROCESSOR INITIALIZED SUCCESSFULLY")
    print(f"📊 Field count: {hybrid_processor.field_count}")
    print(f"🎯 Model path: {hybrid_processor.model_path}")
    print(f"🧹 ExtractionCleaner: {'✅ Active' if hybrid_processor.cleaner else '❌ Missing'}")
    print(f"🔧 Device: {hybrid_processor.device}")
    print(f"🚀 Model loaded: {'✅ Yes' if hybrid_processor.model else '❌ No'}")
    
    # Additional model info
    model_info = hybrid_processor.get_model_info()
    print(f"📋 Model info: {model_info}")
    
except Exception as e:
    print(f"❌ Initialization failed: {e}")
    import traceback
    traceback.print_exc()
    
    # Provide helpful debugging information
    print("\n🔍 DEBUGGING INFO:")
    print(f"   Current working directory: {os.getcwd()}")
    print(f"   Python path: {sys.path[0]}")
    
    # Check for required imports
    try:
        import torch
        print(f"   PyTorch: ✅ {torch.__version__}")
        print(f"   CUDA available: {'✅' if torch.cuda.is_available() else '❌'}")
        if torch.cuda.is_available():
            print(f"   GPU device: {torch.cuda.get_device_name()}")
    except ImportError:
        print("   PyTorch: ❌ Not available")
    
    try:
        from transformers import AutoModel
        print("   Transformers: ✅")
    except ImportError:
        print("   Transformers: ❌ Not available")
    
    raise

🚀 Testing hybrid processor initialization...
🎯 InternVL3 Hybrid processor initialized for 3 fields: DOCUMENT_TYPE → TOTAL_AMOUNT
🔧 CUDA memory allocation configured: max_split_size_mb:64
💡 Using 64MB memory blocks to reduce fragmentation
📊 Initial CUDA state (Multi-GPU Total): Allocated=0.00GB, Reserved=0.00GB
🤖 Auto-detected batch size: 8 (GPU Memory: 279.4GB)
🎯 DOCUMENT AWARE REDUCTION: 3 fields (~90% fewer than original 29)
🎯 Generation config: max_new_tokens=600, temperature=0.0, do_sample=False
🔄 Loading InternVL3 model from: /home/jovyan/nfs_share/models/InternVL3-8B
FlashAttention2 is not installed.


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

✅ InternVL3 model loaded successfully
🔧 Device: cuda:0
💾 Model parameters: 7,944,373,760
🚀 V100 optimizations applied

✅ HYBRID PROCESSOR INITIALIZED SUCCESSFULLY
📊 Field count: 3
🎯 Model path: /home/jovyan/nfs_share/models/InternVL3-8B
🧹 ExtractionCleaner: ✅ Active
🔧 Device: cuda
🚀 Model loaded: ✅ Yes
📋 Model info: {'status': 'loaded', 'model_path': '/home/jovyan/nfs_share/models/InternVL3-8B', 'device': 'cuda:0', 'is_8b_model': True, 'field_count': 3, 'parameter_count': 7944373760}


## Phase 2: Prompt Testing

In [5]:
# Test prompt loading (should use working 82% prompts)
print("📝 Testing prompt loading...")

try:
    # Test different document types
    for doc_type in ['invoice', 'receipt', 'bank_statement']:
        prompt = hybrid_processor.get_extraction_prompt(doc_type)
        print(f"\n📋 {doc_type.upper()} PROMPT:")
        print(f"  📏 Length: {len(prompt)} characters")
        print(f"  🔍 Preview: {prompt[:100]}...")
        
        # Check for working baseline characteristics (simple format)
        if "Extract structured data" in prompt:
            print(f"  ✅ Working baseline format detected")
        else:
            print(f"  ⚠️ May not be using working baseline prompts")
            
except Exception as e:
    print(f"❌ Prompt loading failed: {e}")
    traceback.print_exc()

📝 Testing prompt loading...
📝 Loading invoice prompt for InternVL3 Hybrid

📋 INVOICE PROMPT:
  📏 Length: 2553 characters
  🔍 Preview: You are an expert document analyzer specializing in invoice extraction.
Extract structured data from...
  ✅ Working baseline format detected
📝 Loading receipt prompt for InternVL3 Hybrid

📋 RECEIPT PROMPT:
  📏 Length: 2799 characters
  🔍 Preview: You are an expert document analyzer specializing in receipt extraction.
Extract structured data from...
  ✅ Working baseline format detected
📝 Loading bank_statement prompt for InternVL3 Hybrid

📋 BANK_STATEMENT PROMPT:
  📏 Length: 2768 characters
  🔍 Preview: You are an expert document analyzer specializing in bank statement extraction.
Extract structured da...
  ✅ Working baseline format detected


## Phase 3: Single Image Test (Critical Test)

In [6]:
# Find available test images
print("🔍 Looking for test images...")

# Common test image locations
possible_paths = [
    "evaluation_data/",
    "test_images/",
    "images/",
    "data/"
]

test_images = []
for base_path in possible_paths:
    if os.path.exists(base_path):
        for ext in ['*.jpg', '*.jpeg', '*.png', '*.pdf']:
            import glob
            images = glob.glob(os.path.join(base_path, '**', ext), recursive=True)
            test_images.extend(images[:3])  # Limit to 3 per type

if test_images:
    print(f"📸 Found {len(test_images)} test images:")
    for i, img in enumerate(test_images[:5]):
        print(f"  {i+1}. {img}")
    
    # Use the first available image
    TEST_IMAGE_PATH = test_images[0]
    print(f"\n🎯 Using test image: {TEST_IMAGE_PATH}")
else:
    print("⚠️ No test images found. Please check image paths.")
    print("📂 Current directory contents:")
    import os
    for item in os.listdir('.')[:10]:
        print(f"  - {item}")

🔍 Looking for test images...
⚠️ No test images found. Please check image paths.
📂 Current directory contents:
  - .git
  - .gitignore
  - internvl3_document_aware.py
  - llama_document_aware.py
  - legacy_yaml_backups
  - common
  - README.md
  - llama_document_aware_batch.ipynb
  - llama_document_aware.ipynb
  - notebooks


In [7]:
# CRITICAL TEST: Single image processing
# HARDCODED test image path
TEST_IMAGE_PATH = "/home/jovyan/nfs_share/tod/evaluation_data/image_001.png"

if os.path.exists(TEST_IMAGE_PATH):
    print(f"🧪 CRITICAL TEST: Processing {TEST_IMAGE_PATH}")
    print("="*80)
    
    try:
        start_time = time.time()
        
        # Process single image with hybrid processor
        result = hybrid_processor.process_single_image(TEST_IMAGE_PATH)
        
        processing_time = time.time() - start_time
        
        print("\n🎉 HYBRID PROCESSOR SUCCESS!")
        print("="*80)
        print(f"⏱️ Processing time: {processing_time:.2f} seconds")
        print(f"📊 Fields extracted: {result['extracted_fields_count']}/{result['field_count']}")
        print(f"📈 Response completeness: {result['response_completeness']:.1%}")
        
        print("\n📋 EXTRACTED DATA:")
        for field, value in result['extracted_data'].items():
            status = "✅" if value != "NOT_FOUND" else "❌"
            print(f"  {status} {field}: {value}")
            
        print(f"\n📄 Raw response length: {len(result['raw_response'])} chars")
        
        # Check for ExtractionCleaner evidence
        if "🧹" in str(result) or any("🧹" in str(v) for v in result.values()):
            print("\n🧹 CLEANER ACTIVITY DETECTED - ExtractionCleaner is working!")
        
    except Exception as e:
        print(f"❌ CRITICAL TEST FAILED: {e}")
        import traceback
        traceback.print_exc()
        
else:
    print(f"⚠️ Test image not found at: {TEST_IMAGE_PATH}")
    print("💡 Check that the image file exists")

🧪 CRITICAL TEST: Processing /home/jovyan/nfs_share/tod/evaluation_data/image_001.png
🧹 Memory state: Allocated=14.80GB, Reserved=14.80GB, Fragmentation=0.00GB
📝 Loading invoice prompt for InternVL3 Hybrid
📝 Generated prompt for 3 fields
   Fields: ['DOCUMENT_TYPE', 'SUPPLIER_NAME', 'TOTAL_AMOUNT']
🔍 DOCUMENT-AWARE PROMPT (2553 chars):
You are an expert document analyzer specializing in invoice extraction.
Extract structured data from this invoice image using proven step-by-step methodology.

CRITICAL EXTRACTION RULES:
- If ANY field is not clearly visible in the document, return "NOT_FOUND"
- Do NOT guess, estimate, or infer missing information
- Only extract what is EXPLICITLY shown in the image
- Use exact text from document (preserve original formatting/capitalization)

## STEP-BY-STEP EXTRACTION:

### STEP 1: Document Identification
Look at the header - is it an INVOICE, BILL, QUOTE, or ESTIMATE?
DOCUMENT_TYPE: [INVOICE or NOT_FOUND]

### STEP 2: Business Information (Usually at th

Traceback (most recent call last):
  File "/home/jovyan/nfs_share/tod/LMM_POC/models/document_aware_internvl3_hybrid_processor.py", line 489, in process_single_image
    response = self._resilient_generate(pixel_values, question, **generation_config)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/jovyan/nfs_share/tod/LMM_POC/models/document_aware_internvl3_hybrid_processor.py", line 374, in _resilient_generate
    return self.model.chat(
           ^^^^^^^^^^^^^^^^
  File "/home/jovyan/.cache/huggingface/modules/transformers_modules/InternVL3-8B/modeling_internvl_chat.py", line 291, in chat
    generation_output = self.generate(
                        ^^^^^^^^^^^^^^
  File "/home/jovyan/.conda/envs/unified_vision_processor/lib/python3.11/site-packages/torch/utils/_contextlib.py", line 116, in decorate_context
    return func(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^
  File "/home/jovyan/.cache/huggingface/modules/transformers

## Phase 4: Comparison Test (Optional)

In [8]:
# Compare with current InternVL3 handler (if available)
print("🔄 COMPARISON TEST: Hybrid vs Current InternVL3 Handler")
print("="*80)

try:
    # Try to import current handler for comparison
    from internvl3_document_aware_handler import DocumentAwareInternVL3Handler
    
    # This would require setting up the current handler properly
    # For now, just show that the hybrid processor is working
    print("✅ Current handler available for comparison")
    print("💡 Manual comparison recommended:")
    print("   1. Run same image through current handler")
    print("   2. Compare accuracy and cleaner output")
    print("   3. Check for 🧹 CLEANER CALLED debug output")
    
except ImportError:
    print("ℹ️ Current handler not available for direct comparison")
    print("🎯 Hybrid processor results above show working implementation")

🔄 COMPARISON TEST: Hybrid vs Current InternVL3 Handler
✅ Current handler available for comparison
💡 Manual comparison recommended:
   1. Run same image through current handler
   2. Compare accuracy and cleaner output
   3. Check for 🧹 CLEANER CALLED debug output


## Phase 5: Medium Field Test (If Basic Test Passes)

In [ ]:
# Test with more fields if basic test succeeded
print("🚀 EXTENDED TEST: Medium field list")
print("="*80)

if 'result' in locals() and result['extracted_fields_count'] > 0:
    print("✅ Basic test passed - proceeding with medium field test")
    
    try:
        # Create processor with medium field list - HARDCODED model path
        medium_processor = DocumentAwareInternVL3HybridProcessor(
            field_list=TEST_FIELDS_MEDIUM,
            model_path=INTERNVL3_MODEL_PATH,  # Pass hardcoded path directly
            debug=True
        )
        
        if 'TEST_IMAGE_PATH' in locals() and os.path.exists(TEST_IMAGE_PATH):
            print(f"🧪 Processing with {len(TEST_FIELDS_MEDIUM)} fields...")
            
            medium_result = medium_processor.process_single_image(TEST_IMAGE_PATH)
            
            print(f"\n📊 Medium test results:")
            print(f"  Fields extracted: {medium_result['extracted_fields_count']}/{medium_result['field_count']}")
            print(f"  Response completeness: {medium_result['response_completeness']:.1%}")
            print(f"  Processing time: {medium_result['processing_time']:.2f}s")
            
            # Show additional fields found
            print("\n📋 Additional fields (vs small test):")
            for field in TEST_FIELDS_MEDIUM:
                if field not in TEST_FIELDS_SMALL:
                    value = medium_result['extracted_data'].get(field, 'NOT_FOUND')
                    status = "✅" if value != "NOT_FOUND" else "❌"
                    print(f"  {status} {field}: {value}")
        
    except Exception as e:
        print(f"❌ Medium test failed: {e}")
        print("💡 Basic processor still working - issue may be with increased field count")
        
else:
    print("⚠️ Skipping medium test - basic test did not extract any fields")
    print("💡 Focus on debugging basic functionality first")

## Summary and Next Steps

In [10]:
# Test summary
print("🎯 HYBRID PROCESSOR TEST SUMMARY")
print("="*80)

# Check what tests passed
tests_passed = []
tests_failed = []

if 'hybrid_processor' in locals():
    tests_passed.append("✅ Processor initialization")
else:
    tests_failed.append("❌ Processor initialization")

if 'result' in locals() and result.get('extracted_fields_count', 0) > 0:
    tests_passed.append("✅ Single image processing")
    tests_passed.append("✅ Field extraction working")
else:
    tests_failed.append("❌ Single image processing")

if 'medium_result' in locals() and medium_result.get('extracted_fields_count', 0) > 0:
    tests_passed.append("✅ Medium field list processing")

print("\n🟢 PASSED TESTS:")
for test in tests_passed:
    print(f"  {test}")

if tests_failed:
    print("\n🔴 FAILED TESTS:")
    for test in tests_failed:
        print(f"  {test}")

print("\n🚀 NEXT STEPS:")
if len(tests_passed) >= 2:
    print("  1. ✅ Hybrid processor is working!")
    print("  2. 🎯 Test with full document field lists")
    print("  3. 📊 Compare accuracy with baseline (79%/77%/55%)")
    print("  4. 🔍 Look for 🧹 CLEANER CALLED output in verbose mode")
    print("  5. 📈 Target: Restore 82% accuracy baseline")
else:
    print("  1. 🔧 Debug initialization or image loading issues")
    print("  2. 📋 Check model path and GPU availability")
    print("  3. 🖼️ Verify test image format and accessibility")

print("\n💡 HYBRID ARCHITECTURE BENEFITS:")
print("  - InternVL3 model accuracy + Llama processing reliability")
print("  - ExtractionCleaner integration for value normalization")
print("  - Working 82% prompts (simple format)")
print("  - Zero risk to existing Llama pipeline")

🎯 HYBRID PROCESSOR TEST SUMMARY

🟢 PASSED TESTS:
  ✅ Processor initialization

🔴 FAILED TESTS:
  ❌ Single image processing

🚀 NEXT STEPS:
  1. 🔧 Debug initialization or image loading issues
  2. 📋 Check model path and GPU availability
  3. 🖼️ Verify test image format and accessibility

💡 HYBRID ARCHITECTURE BENEFITS:
  - InternVL3 model accuracy + Llama processing reliability
  - ExtractionCleaner integration for value normalization
  - Working 82% prompts (simple format)
  - Zero risk to existing Llama pipeline
